# EarthBridge Kaggle Baseline Training

Use this notebook for free online training on the Kaggle BEN-14K dataset. It expects `/kaggle/input/datasets/narendraaironi/bigearthnet-14k/BEN_14k`, then exports `artifacts/earthbridge_export.zip` for the local demo.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = ""  # Optional: set to your GitHub repo URL.
ROOT = Path("/kaggle/working/EarthBridge")

if REPO_URL and not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

if not ROOT.exists():
    ROOT = Path.cwd()

os.chdir(ROOT)
print("Working directory:", ROOT)

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q faiss-cpu rasterio albumentations opencv-python matplotlib tensorboard fastapi "uvicorn[standard]" python-multipart
!python -m pip install -q -e .

## Configure Dataset

The default path targets Kaggle BEN-14K. `BigEarthNet-S1` is treated as SAR and `BigEarthNet-S2` is treated as multispectral.

In [ ]:
DATA_RAW = Path("/kaggle/input/datasets/narendraaironi/bigearthnet-14k/BEN_14k")
IMAGE_ROOT = DATA_RAW.parent

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DATA_RAW:", DATA_RAW)
print("IMAGE_ROOT:", IMAGE_ROOT)
print("DEVICE:", DEVICE)
assert DATA_RAW.exists(), f"Dataset path does not exist: {DATA_RAW}"

In [ ]:
!python scripts/inspect_data.py --input "{DATA_RAW}" --output-dir artifacts/reports
!python scripts/build_manifest.py --input "{DATA_RAW}" --output data/manifests/samples.csv
!python scripts/create_splits.py --manifest data/manifests/samples.csv --output-dir data/manifests

In [ ]:
!python scripts/train_baseline.py \
  --manifest data/manifests/train.csv \
  --root-dir "{IMAGE_ROOT}" \
  --left-modality multispectral \
  --right-modality sar \
  --image-size 128 \
  --batch-size 16 \
  --epochs 5 \
  --device "{DEVICE}" \
  --output-checkpoint artifacts/checkpoints/baseline_pair.pt

In [ ]:
!python scripts/generate_descriptors.py \
  --manifest data/manifests/test.csv \
  --root-dir "{IMAGE_ROOT}" \
  --checkpoint artifacts/checkpoints/baseline_pair.pt \
  --output-dir artifacts/descriptors \
  --name gallery \
  --image-size 128 \
  --device "{DEVICE}"

!python scripts/build_indexes.py \
  --descriptors artifacts/descriptors/gallery.npy \
  --ids artifacts/descriptors/gallery_ids.json \
  --output-index artifacts/indexes/gallery.index

In [ ]:
!python scripts/evaluate_descriptors.py \
  --manifest data/manifests/test.csv \
  --descriptors artifacts/descriptors/gallery.npy \
  --ids artifacts/descriptors/gallery_ids.json \
  --relevance-mode semantic \
  --output-dir artifacts/reports

!python scripts/benchmark_latency.py \
  --descriptors artifacts/descriptors/gallery.npy \
  --ids artifacts/descriptors/gallery_ids.json \
  --top-k 10 \
  --output artifacts/reports/latency_summary.json

In [ ]:
!mkdir -p artifacts/manifests
!cp data/manifests/test.csv artifacts/manifests/test.csv
!python scripts/verify_artifacts.py --artifact-root artifacts
!python scripts/export_artifacts.py --artifact-root artifacts --output artifacts/earthbridge_export.zip
print("Download this file from Kaggle:", ROOT / "artifacts/earthbridge_export.zip")